# **Import Lib**

In [ ]:
import pandas as pd
import numpy as np

# **Load ViHSD dataset**

In [ ]:
!gdown --id 1Bs-QydHnSEtvqNtSqzHEF_bILZhea_9M

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1Bs-QydHnSEtvqNtSqzHEF_bILZhea_9M
To: /content/ViHSD.zip
100% 914k/914k [00:00<00:00, 14.7MB/s]


In [ ]:
!unzip ViHSD.zip

Archive:  ViHSD.zip
   creating: ViHSD/
  inflating: ViHSD/dev.csv           
  inflating: ViHSD/test.csv          
  inflating: ViHSD/train.csv         


Các label của ViHSD dataset:
+ Clean (0)
+ Offensive (1)
+ Hate (2)

In [ ]:
df_train = pd.read_csv('ViHSD/train.csv')
df_test = pd.read_csv('ViHSD/test.csv')
df_val = pd.read_csv('ViHSD/dev.csv')

In [ ]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24048 entries, 0 to 24047
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   free_text  24046 non-null  object
 1   label_id   24048 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 375.9+ KB


In [ ]:
df_train["label_id"].value_counts()

,count
label_id,
0,19886
2,2556
1,1606


In [ ]:
df_test["label_id"].value_counts()

,count
label_id,
0,5548
2,688
1,444


In [ ]:
df_val["label_id"].value_counts()

,count
label_id,
0,2190
2,270
1,212


In [ ]:
df_full = pd.concat([df_train, df_test, df_val])
df_full["label_id"].value_counts()

,count
label_id,
0,27624
2,3514
1,2262


Với yêu cầu recognition thêm các bài viết / status có yếu tố scam nên phải xây dựng thêm data với nhãn là Scam:
+ Scam (3)

# **Load created Scam Spam Dataset**

In [ ]:
!gdown --id 1XiXCg1AB4tyG4qRe2bFoN9igf6dgkx9T

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1XiXCg1AB4tyG4qRe2bFoN9igf6dgkx9T
To: /content/scam_spam_dataset.zip
100% 37.2k/37.2k [00:00<00:00, 55.6MB/s]


In [ ]:
!unzip scam_spam_dataset.zip

Archive:  scam_spam_dataset.zip
   creating: scam_spam_dataset/
  inflating: scam_spam_dataset/vietnamese_scam_dataset_2000.json  
  inflating: scam_spam_dataset/vietnamese_spam_dataset_2000.json  


In [ ]:
df_scam = pd.read_json("/content/scam_spam_dataset/vietnamese_scam_dataset_2000.json")
df_scam.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    2000 non-null   object
 1   label   2000 non-null   object
dtypes: object(2)
memory usage: 31.4+ KB


In [ ]:
df_scam.head()

,text,label
0,Cơ quan thuế thông báo bạn còn nợ phí kê khai ...,Scam
1,💥 FLASH SALE chỉ còn 30 phút! AirPods Pro giá ...,Scam
2,Shop bên em cần người thả tim sản phẩm TikTokS...,Scam
3,"Hỗ trợ vay nóng 5-100 triệu, không cần thế chấ...",Scam
4,"Phiên bản app ngân hàng mới đã phát hành, tải ...",Scam


In [ ]:
def preprocessing_data(df, label_id):
    df_tmp = df.copy()
    df_tmp.rename(columns={"text": "free_text"}, inplace=True)
    df_tmp["label_id"] = [label_id for _ in range(len(df))]
    df_tmp.drop(columns=["label"], inplace=True)
    return df_tmp

In [ ]:
df_scam = preprocessing_data(df_scam, 3)
df_scam.head()

,free_text,label_id
0,Cơ quan thuế thông báo bạn còn nợ phí kê khai ...,3
1,💥 FLASH SALE chỉ còn 30 phút! AirPods Pro giá ...,3
2,Shop bên em cần người thả tim sản phẩm TikTokS...,3
3,"Hỗ trợ vay nóng 5-100 triệu, không cần thế chấ...",3
4,"Phiên bản app ngân hàng mới đã phát hành, tải ...",3


In [ ]:
# df_spam = pd.read_json("/content/scam_spam_dataset/vietnamese_spam_dataset_2000.json")

In [ ]:
# df_spam = preprocessing_data(df_spam, 4)
# df_spam.head()

In [ ]:
# df_full = pd.concat([df_full, df_scam, df_spam])
df_full = pd.concat([df_full, df_scam])
df_full["label_id"].value_counts()

,count
label_id,
0,27624
2,3514
1,2262
3,2000


In [ ]:
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(df_full, test_size=0.28, random_state=42, stratify=df_full["label_id"])
df_test, df_val = train_test_split(df_test, test_size=0.28, random_state=42, stratify=df_test["label_id"])

In [ ]:
df_train["label_id"].value_counts()

,count
label_id,
0,19888
2,2530
1,1629
3,1440


In [ ]:
df_test["label_id"].value_counts()

,count
label_id,
0,5570
2,708
1,456
3,403


In [ ]:
df_val["label_id"].value_counts()

,count
label_id,
0,2166
2,276
1,177
3,157


In [ ]:
df_train.to_csv("/content/drive/MyDrive/CyberSoft_Courses/ML_DL_04/final_capstone/combine_dataset/train.csv")
df_test.to_csv("/content/drive/MyDrive/CyberSoft_Courses/ML_DL_04/final_capstone/combine_dataset/test.csv")
df_val.to_csv("/content/drive/MyDrive/CyberSoft_Courses/ML_DL_04/final_capstone/combine_dataset/val.csv")